# 🧬 Pipeline scRNA-seq — IBD Colon/Recto - GSE214695
## Descarga de datos

# · Preparación del entorno

In [ ]:
# Montar Google Drive y prepara el gestor de entornos Conda,
# que aísla las dependencias del proyecto del entorno base de Colab.
from google.colab import drive
drive.mount('/content/drive')

!pip install -q condacolab
import condacolab
condacolab.install()

Mounted at /content/drive
⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.11.0-1/Miniforge3-25.11.0-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:14
🔁 Restarting kernel...


In [ ]:
# Instalación del entorno Conda del proyecto (environment.yml), con las
# versiones exactas de todas las dependencias fijadas para reproducibilidad.

!conda env update -n base -f /content/drive/MyDrive/TFM_IBD_GSE214695/environment.yml -q

Retrieving notices: ...working... done
Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: ...working... done
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done
Installing pip dependencies: ...working... done


In [ ]:
import scanpy as sc
import pandas as pd
import os
import tarfile
import tempfile
import warnings
import gc # Para liberar RAM
import urllib.request
import yaml
from pathlib import Path
import yaml

warnings.filterwarnings("ignore", category=DeprecationWarning)

/usr/local/lib/python3.12/site-packages/scanpy/_utils/__init__.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/usr/local/lib/python3.12/site-packages/scanpy/__init__.py:36: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/usr/local/lib/python3.12/site-packages/scanpy/readwrite.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


# · Cargar los datos y rutas

In [ ]:
REPO_ROOT = Path("/content/drive/MyDrive/TFM_IBD_GSE214695")
with open(REPO_ROOT / "config" / "params.yaml") as f:
    PARAMS = yaml.safe_load(f)

PROJECT_ROOT = os.environ.get(PARAMS["project_root_env_var"], PARAMS["project_root_default"])

data_dir      = f"{PROJECT_ROOT}/{PARAMS['paths']['raw']['tar_file']}"
metadata_file = f"{PROJECT_ROOT}/{PARAMS['paths']['raw']['annotation_file']}"
processed_dir = f"{PROJECT_ROOT}/{PARAMS['paths']['interim']['00_raw_h5ad']}/"

Path(processed_dir).mkdir(parents=True, exist_ok=True)

In [ ]:
GEO_TAR_URL = "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE214695&format=file"
GEO_ANNOTATION_URL = (
    "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE214695&format=file&file=GSE214695%5Fcell%5Fannotation%2Ecsv%2Egz"
)

Path(data_dir).parent.mkdir(parents=True, exist_ok=True)

if not os.path.exists(data_dir):
    print(f"Descargando {data_dir} ...")
    urllib.request.urlretrieve(GEO_TAR_URL, data_dir)
else:
    print(f"Ya existe, se omite descarga: {data_dir}")

if not os.path.exists(metadata_file):
    print(f"Descargando {metadata_file} ...")
    urllib.request.urlretrieve(GEO_ANNOTATION_URL, metadata_file)
else:
    print(f"Ya existe, se omite descarga: {metadata_file}")

Ya existe, se omite descarga: /content/drive/MyDrive/IBD_TFM/data/raw/GSE214695_RAW.tar
Ya existe, se omite descarga: /content/drive/MyDrive/IBD_TFM/data/raw/GSE214695_cell_annotation.csv.gz


# Creación de los archivos .h5ad

In [ ]:
# Abrimos el archivo comprimido (tar) para leer su contenido

with tarfile.open(data_dir, "r") as tar:

    # Buscamos todos los archivos que contienen la palabra 'matrix.mtx'

    all_files = tar.getnames()
    mtx_files = [f for f in all_files if 'matrix.mtx' in f]

    # Extraemos el "prefijo" para saber qué archivos van juntos en cada muestra

    prefijos = sorted([f.split('matrix.mtx')[0] for f in mtx_files])
    print(f"Muestras detectadas: {len(prefijos)}")

    h5ad_files = [] # Almacena las rutas de salida

    # Creamos un directorio temporal para extraer los archivos sin llenar el sistema

    with tempfile.TemporaryDirectory() as tmp_dir:

        # Leer y unir una por una las muestras

        for p in prefijos:

            name_id = p.strip('_')
            output_path = os.path.join(processed_dir, f"{name_id}.h5ad")
            print(f"Procesando: {name_id}")

            # Localiza los 3 archivos que componen las muestras

            f_mtx = [f for f in all_files if f.startswith(p) and 'matrix.mtx' in f][0]
            f_var = [f for f in all_files if f.startswith(p) and 'features.tsv' in f][0]
            f_obs = [f for f in all_files if f.startswith(p) and 'barcodes.tsv' in f][0]

            # Extrae los archivos necesarios y los lleva al directorio temporal

            tar.extract(f_mtx, path=tmp_dir)
            tar.extract(f_var, path=tmp_dir)
            tar.extract(f_obs, path=tmp_dir)

            # Carga de la matriz de la muestra actual y se traspone (.T) para que las filas sean células y las columnas genes.
            adata = sc.read_mtx(os.path.join(tmp_dir, f_mtx)).T

            # Asignación de nombres de genes (variables)

            genes = pd.read_csv(os.path.join(tmp_dir, f_var), sep='\t', header=None)
            adata.var_names = genes[1].values
            # Añadimos sufijos en caso de que se repita el nombre de algún gen
            adata.var_names_make_unique()

            # Identificación de células añadiendo el ID de la muestra como prefijo

            barcodes = pd.read_csv(os.path.join(tmp_dir, f_obs), sep='\t', header=None)
            adata.obs_names = [f"{name_id}_{code}" for code in barcodes[0]]
            # Guardamos el id de la celula para filtrar por muestra cuando sea necesario
            adata.obs['sample_id'] = name_id

            # Guardar muestra individual en formato .h5ad y liberar RAM
            adata.write(output_path)
            h5ad_files.append(output_path)

            # Borramos la muestra individual ya procesada para liberar memoria
            del adata
            gc.collect()

Muestras detectadas: 18
Procesando: GSM6614348_HC-1
Procesando: GSM6614349_HC-2
Procesando: GSM6614350_HC-3
Procesando: GSM6614351_HC-4
Procesando: GSM6614352_HC-5
Procesando: GSM6614353_HC-6
Procesando: GSM6614354_UC-1
Procesando: GSM6614355_UC-2
Procesando: GSM6614356_UC-3
Procesando: GSM6614357_UC-4
Procesando: GSM6614358_UC-5
Procesando: GSM6614359_UC-6
Procesando: GSM6614360_CD-1
Procesando: GSM6614361_CD-2
Procesando: GSM6614362_CD-3
Procesando: GSM6614363_CD-4
Procesando: GSM6614364_CD-5
Procesando: GSM6614365_CD-6


# Comprobación del contenido de los archivos

In [ ]:
# Carga de una muestra como verificación del correcto contenido de los archivos
adata = sc.read_h5ad(f"{processed_dir}GSM6614353_HC-6.h5ad")

print("-- VERIFICACIÓN DE DATOS --")
# Confirmar el número de genes y células.
print(f"Dimensiones (Células x Genes): {adata.shape}")

# Comprobamos que la suma de valores de la matriz no es 0 o Na.
print(f"Suma total de conteos en el objeto: {adata.X.sum()}")

# Verficar que las columnas contienen nombres de genes
print("\nPrimeros 5 genes (Features):")
print(adata.var_names[:5].tolist()) # .tolist convierte el índice en una lista simple

# Comprobar una parte de la matriz de datos
print("\nExtracto de la matriz de datos (Primeras 3 células/genes):")
print(adata.X[:3, :3].toarray()) # .toarray() si es matriz dispersa

# Confirmar que el prefijo de la muestra se aplicó a los códigos de barras de las células
print("Primeros 5 barcodes (obs_names):")
print(adata.obs_names[:5].tolist())


-- VERIFICACIÓN DE DATOS --
Dimensiones (Células x Genes): (6794880, 33538)
Suma total de conteos en el objeto: 32213884.0

Primeros 5 genes (Features):
['MIR1302-2HG', 'FAM138A', 'OR4F5', 'AL627309.1', 'AL627309.3']

Extracto de la matriz de datos (Primeras 3 células/genes):
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
Primeros 5 barcodes (obs_names):
['GSM6614353_HC-6_AAACCCAAGAAACACT-1', 'GSM6614353_HC-6_AAACCCAAGAAACCAT-1', 'GSM6614353_HC-6_AAACCCAAGAAACCCA-1', 'GSM6614353_HC-6_AAACCCAAGAAACCCG-1', 'GSM6614353_HC-6_AAACCCAAGAAACCTG-1']
